# DynamoDB Design Patterns

## Overview
DynamoDB design is fundamentally different from relational design. The core principle:

> **Design your data model around your access patterns, not your entities.**

In SQL, you normalise your schema and add indexes as queries emerge. In DynamoDB, you must know your access patterns upfront because the partition key, sort key, and indexes are fixed at table creation.

**The design process:**
1. **List all access patterns** — every query the application will make
2. **Group patterns by partition** — what is the natural grouping unit?
3. **Choose the sort key** — what ordering or range query is needed within a partition?
4. **Identify patterns not served by the base key** — these need GSIs
5. **Decide: single table or multiple tables** — based on complexity and team experience

**Single-table design:**
All entity types (sites, observations, water quality, species) in one table using generic key names (`PK`, `SK`) with entity-type prefixes. This minimises the number of requests needed to fetch related data and reduces operational overhead.

**When multiple tables are better:**
- Different teams own different entity types
- Access patterns are completely separate
- Team is not yet experienced with single-table design

---

In [ ]:

print("=== Access pattern analysis: the starting point ===")
print()
print("For the ecological monitoring system, list all required access patterns FIRST:")
access_patterns = [
    ("AP-01", "Get site metadata",                         "GetItem: PK=SITE#x, SK=META#SITE"),
    ("AP-02", "List all active sites",                     "Scan (small table) or GSI on active flag"),
    ("AP-03", "All observations at a site",                "Query: PK=SITE#x, SK begins_with OBS#"),
    ("AP-04", "Observations at site in date range",        "Query: PK=SITE#x, SK between OBS#date1 and OBS#date2"),
    ("AP-05", "One specific observation",                  "GetItem: PK=SITE#x, SK=OBS#date#id"),
    ("AP-06", "All observations of a species (any site)",  "GSI query: PK=species_id, SK begins_with OBS#"),
    ("AP-07", "Water quality for a site",                  "Query: PK=SITE#x, SK begins_with WQ#"),
    ("AP-08", "Water quality in date range",               "Query: PK=SITE#x, SK between WQ#date1 and WQ#date2"),
    ("AP-09", "All at-risk species observations",          "GSI on at_risk flag + Query (sparse index)"),
    ("AP-10", "All observations by crew member",           "GSI query: PK=crew_id, SK begins_with OBS#"),
    ("AP-11", "Count of observations per site",            "Maintained as an attribute on META#SITE item"),
    ("AP-12", "Species observed at multiple sites",        "GSI on species_id; scan results in application"),
]
print(f"  {'ID':<7s}  {'Access Pattern':<44s}  DynamoDB Strategy")
print("  " + "-"*90)
for ap_id, pattern, strategy in access_patterns:
    print(f"  {ap_id:<7s}  {pattern:<44s}  {strategy}")


---
## Single-table design

In [ ]:

print("=== Single-table design: all entities in one table ===")
print()
single_table = '''
# Single-table design puts multiple entity types (sites, observations,
# water quality, species) into ONE table using a generic PK/SK schema.
#
# KEY INSIGHT: different entity types use different SK prefixes.
# This allows efficient queries per entity type without separate tables.

# ──────────────────────────────────────────────────────────────────
# Table: EcoMonitoring  (single table for all entities)
# PK (site_id / entity id)    SK (entity type + date + id)
# ──────────────────────────────────────────────────────────────────
#
# Site metadata:
{"PK": "SITE#001", "SK": "META#SITE",
 "site_name": "Fundy Estuary", "region": "Atlantic", "active": True}
#
# Species record (stored under site that first documented it):
{"PK": "SPECIES#SALMON", "SK": "META#SPECIES",
 "common_name": "Atlantic Salmon", "scientific": "Salmo salar",
 "at_risk": True, "taxon": "Fish"}
#
# Observation at site:
{"PK": "SITE#001", "SK": "OBS#2023-04-10#001",
 "species_id": "SPECIES#SALMON", "count": 12, "method": "Electrofishing",
 "crew_id": "CREW#001", "GSI1PK": "SPECIES#SALMON", "GSI1SK": "OBS#2023-04-10#001"}
#
# Water quality at site:
{"PK": "SITE#001", "SK": "WQ#2023-04-10",
 "ph": "7.2", "dissolved_o2": "9.1", "temp_c": "8.5"}
#
# Crew member:
{"PK": "CREW#001", "SK": "META#CREW",
 "full_name": "Dr. Aroha Ngata", "role": "Biologist", "certified": True}
#
# Crew-to-observation (for AP-10: all obs by crew member):
{"PK": "CREW#001", "SK": "OBS#2023-04-10#001",
 "site_id": "SITE#001", "GSI1PK": "CREW#001", "GSI1SK": "OBS#2023-04-10#001"}
#
# Queries enabled by this design:
# AP-01: GetItem PK=SITE#001, SK=META#SITE
# AP-03: Query PK=SITE#001, SK begins_with OBS#
# AP-07: Query PK=SITE#001, SK begins_with WQ#
# AP-06: GSI1 Query PK=SPECIES#SALMON, SK begins_with OBS#
# AP-10: GSI1 Query PK=CREW#001, SK begins_with OBS#
'''
print(single_table)


---
## Overloaded GSI and adjacency list patterns

In [ ]:

print("=== Overloaded GSI and adjacency list patterns ===")
overloaded = '''
# ── Overloaded GSI (GSI1PK / GSI1SK) ──────────────────────────────
# Use generic GSI key names (GSI1PK, GSI1SK) so many entity types
# can share the same GSI, each using it for a different purpose.
#
# Species observations:  GSI1PK=SPECIES#SALMON, GSI1SK=OBS#2023-04-10#001
# Crew observations:     GSI1PK=CREW#001,       GSI1SK=OBS#2023-04-10#001
# Region sites:          GSI1PK=REGION#ATLANTIC, GSI1SK=SITE#001
#
# One GSI serves three completely different access patterns.
# Without overloading, you would need 3 GSIs.

# ── Adjacency list (many-to-many) ────────────────────────────────
# Sites can have multiple species; species can appear at multiple sites.
# Model as two item types: one per site-species relationship direction.

# Forward: site → species observed
{"PK": "SITE#001", "SK": "SPECIES#SALMON",
 "first_observed": "2023-04-10", "total_count": 48}

# Reverse: species → sites where observed
{"PK": "SPECIES#SALMON", "SK": "SITE#001",
 "first_observed": "2023-04-10", "total_count": 48}

# Query "all species at SITE#001":       PK=SITE#001, SK begins_with SPECIES#
# Query "all sites for SPECIES#SALMON":  PK=SPECIES#SALMON, SK begins_with SITE#
# No JOIN needed; both directions stored as items.
'''
print(overloaded)

print("=== Counter pattern: atomic counters ===")
counter = '''
# Track observation count per site atomically:
# Each new observation updates the site metadata item:
table.update_item(
    Key={"PK": "SITE#001", "SK": "META#SITE"},
    UpdateExpression="ADD obs_count :one, species_set :species",
    ExpressionAttributeValues={
        ":one":     1,
        ":species": {"SPECIES#SALMON"},   # NS = number set; SS = string set
    }
)
# obs_count: atomic integer increment (safe under concurrent writes)
# species_set: atomic add to string set (automatically deduplicated)

# Read the counter:
response = table.get_item(
    Key={"PK": "SITE#001", "SK": "META#SITE"},
    ProjectionExpression="obs_count, species_set"
)
print(response["Item"])
# {"obs_count": 32, "species_set": {"SPECIES#SALMON", "SPECIES#HERON", ...}}
'''
print(counter)


---
## Common Pitfalls

**1. Jumping straight to single-table design without access pattern analysis**
Single-table design is powerful but complex. Beginners who adopt it without fully mapping access patterns end up with a tangled key schema that is hard to query and impossible to extend. Start by listing every access pattern. If the table serves 3-5 clear patterns, single-table may be right. If patterns are still being discovered, use multiple tables first.

**2. Using sequential IDs or timestamps as partition keys**
A partition key of `date = '2023-04-10'` means all writes on that date go to one partition -- a severe hot partition for any write-heavy table. Add a random suffix or hash to distribute writes: `date_shard = '2023-04-10#3'` (suffix 0-N based on hash). Alternatively, make the high-cardinality entity ID (site_id, user_id) the partition key and date the sort key.

**3. Putting too many entity types in a single-table without careful GSI planning**
Single-table designs rely on GSIs to support access patterns across entity types. If you put 10 entity types in one table with 3 GSIs, not all access patterns are efficiently served. Model complexity grows non-linearly. Multiple tables are sometimes simpler and more maintainable.

**4. Forgetting to write both sides of an adjacency list**
An adjacency list requires two writes per relationship: one forward item and one reverse item. If one write fails (network error, Lambda timeout), the relationship is asymmetric -- one direction returns results, the other does not. Use TransactWriteItems to keep both sides atomic.

**5. Overloaded GSI keys making queries ambiguous**
When GSI1PK holds `SPECIES#SALMON` for one entity type and `CREW#001` for another, a stale or corrupted item with the wrong GSI1PK value will appear in the wrong GSI query results. Always validate GSI key values before writing and test each query pattern against realistic data.

**6. Unbounded counter growth in SET attributes**
`ADD species_set :new_species` is great for small sets but if a site eventually observes thousands of species, the species_set grows without bound. DynamoDB limits item size to 400 KB including all attributes. Monitor item size with `ITEM_SIZE` in responses and split large set attributes into separate items when they approach a few hundred entries.


---
*sql_methods_library - Samantha McGarrigle*